# Quickstart: Downloading and parsing a real MaveDB score set

This notebook shows how to search MaveDB for a real score set, download its score table, and convert it into a standardized dataset using `dms_parser`.

We will:

1. Search public MaveDB score sets
2. Select one score set by URN
3. Download its metadata
4. Download its scores table as CSV
5. Identify the relevant columns
6. Provide or recover the WT sequence
7. Build a standardized dataset with `dms_parser`

In [1]:
from pathlib import Path
import requests
import pandas as pd

from dms_parser import build_mavedb_dataset, read_table, write_table

## 1. Define local directories

In [10]:
DATA_DIR = Path("data/mavedb")
DATA_DIR.mkdir(parents=True, exist_ok=True)

## 2. Search public MaveDB score sets

MaveDB exposes a public REST API. Public reads do not require authentication.
The search endpoint uses a POST request with a JSON body.

In [2]:
BASE_URL = "https://api.mavedb.org/api/v1"
search_text = "UBE2I"

response = requests.post(
    f"{BASE_URL}/score-sets/search",
    json={"text": search_text},
    timeout=60,
)
response.raise_for_status()

search_results = response.json()
len(search_results)

2

In [3]:
df_search = pd.DataFrame(search_results)
df_search.head()

,scoreSets,numScoreSets
0,"{'urn': 'urn:mavedb:00000001-c-2', 'title': 'H...",10
1,"{'urn': 'urn:mavedb:00000001-c-1', 'title': 'H...",10
2,"{'urn': 'urn:mavedb:00000001-b-1', 'title': 'S...",10
3,"{'urn': 'urn:mavedb:00000001-b-2', 'title': 'S...",10
4,"{'urn': 'urn:mavedb:00000001-d-2', 'title': 'T...",10


In [5]:
if "scoreSets" in df_search.columns:
    df_search = pd.json_normalize(df_search["scoreSets"])

df_search.head()

,urn,title,shortDescription,publishedDate,replacesId,numVariants,primaryPublicationIdentifiers,secondaryPublicationIdentifiers,creationDate,modificationDate,...,experiment.numScoreSets,experiment.processingState,experiment.officialCollections,license.longName,license.shortName,license.active,license.link,license.version,license.id,license.recordType
0,urn:mavedb:00000001-c-2,Human Calmodulin imputed and refined,A machine-learning imputed and refined Deep Mu...,2018-06-29,None,2980,"[{'identifier': '29269382', 'dbName': 'PubMed'...",[],2018-06-29,2024-11-26,...,2,None,[],CC0 (Public domain),CC0,True,https://creativecommons.org/publicdomain/zero/...,1.0,1,ShortLicense
1,urn:mavedb:00000001-c-1,Human Calmodulin DMS-TileSeq,A Deep Mutational Scan of human Calmodulin usi...,2018-06-29,None,5957,"[{'identifier': '29269382', 'dbName': 'PubMed'...",[],2018-06-29,2025-10-03,...,2,None,[],CC0 (Public domain),CC0,True,https://creativecommons.org/publicdomain/zero/...,1.0,1,ShortLicense
2,urn:mavedb:00000001-b-1,SUMO1 imputed and refined,A machine-learning imputed and refined Deep Mu...,2018-06-29,None,2020,"[{'identifier': '29269382', 'dbName': 'PubMed'...",[],2018-06-29,2024-11-26,...,2,None,[],CC0 (Public domain),CC0,True,https://creativecommons.org/publicdomain/zero/...,1.0,1,ShortLicense
3,urn:mavedb:00000001-b-2,SUMO1 DMS-TileSeq,A Deep Mutational Scan of the human SUMO1 usin...,2018-06-29,None,4698,"[{'identifier': '29269382', 'dbName': 'PubMed'...",[],2018-06-29,2024-11-26,...,2,None,[],CC0 (Public domain),CC0,True,https://creativecommons.org/publicdomain/zero/...,1.0,1,ShortLicense
4,urn:mavedb:00000001-d-2,TPK1 imputed and refined,A machine-learning imputed and refined Deep Mu...,2018-06-29,None,4860,"[{'identifier': '29269382', 'dbName': 'PubMed'...",[],2018-06-29,2024-11-27,...,1,None,[],CC0 (Public domain),CC0,True,https://creativecommons.org/publicdomain/zero/...,1.0,1,ShortLicense


## 3. Inspect search results

We usually care about:
- `urn`
- `title`
- `numVariants`
- any target-related fields if present

In [6]:
candidate_cols = [c for c in ["urn", "title", "numVariants", "createdAt", "updatedAt"] if c in df_search.columns]
df_search[candidate_cols].head(10)

,urn,title,numVariants
0,urn:mavedb:00000001-c-2,Human Calmodulin imputed and refined,2980
1,urn:mavedb:00000001-c-1,Human Calmodulin DMS-TileSeq,5957
2,urn:mavedb:00000001-b-1,SUMO1 imputed and refined,2020
3,urn:mavedb:00000001-b-2,SUMO1 DMS-TileSeq,4698
4,urn:mavedb:00000001-d-2,TPK1 imputed and refined,4860
5,urn:mavedb:00001251-a-1,TPK1 DMS-TileSeq,9542
6,urn:mavedb:00000001-a-1,UBE2I imputed & refined,3180
7,urn:mavedb:00000001-a-3,UBE2I DMS-TileSeq,7390
8,urn:mavedb:00000001-a-4,UBE2I joint data,10438
9,urn:mavedb:00000001-a-2,UBE2I DMS-BarSeq,3418


In [7]:
selected = df_search.iloc[0]
score_set_urn = selected["urn"]

print("Selected URN:", score_set_urn)
print("Selected title:", selected.get("title"))

Selected URN: urn:mavedb:00000001-c-2
Selected title: Human Calmodulin imputed and refined


## 5. Download score set metadata

MaveDB provides a score-set metadata endpoint identified by URN.

In [8]:
meta_response = requests.get(f"{BASE_URL}/score-sets/{score_set_urn}", timeout=60)
meta_response.raise_for_status()

score_set_meta = meta_response.json()
score_set_meta.keys()

dict_keys(['title', 'methodText', 'abstractText', 'shortDescription', 'extraMetadata', 'recordType', 'urn', 'numVariants', 'license', 'metaAnalyzesScoreSetUrns', 'metaAnalyzedByScoreSetUrns', 'doiIdentifiers', 'primaryPublicationIdentifiers', 'secondaryPublicationIdentifiers', 'publishedDate', 'creationDate', 'modificationDate', 'createdBy', 'modifiedBy', 'targetGenes', 'datasetColumns', 'externalLinks', 'contributors', 'scoreCalibrations', 'experiment', 'officialCollections', 'private', 'processingState', 'mappingState'])

In [9]:
score_set_meta

{'title': 'Human Calmodulin imputed and refined',
 'methodText': '##Scoring procedure:\r\nDMS-TileSeq reads were processed using the [dmsPipeline](https://bitbucket.org/rothlabto/dmspipeline) software. Briefly, TileSeq read counts were used to establish relative allele frequencies in each condition. Non-mutagenized control counts were subtracted from counts (as estimates of sequencing error). log ratios of selection over non-selection counts were calculated. The resulting TileSeq fitness values were then normalized to 0-1 scale where 0 corresponds to the median nonsense score and 1 corresponds to the median synonymous score. Random-Forest-based machine learning was used to impute missing values and refine low-confidence measurements, based on intrinsic, structural, and biochemical features.\r\n\r\nSee [**Weile *et al.* 2017**](http://msb.embopress.org/content/13/12/957) for more details.\r\n\r\n## Additional columns:\r\n* exp.score = experimental score from the joint DMS-BarSeq/DMS-Til

## 6. Download the score table as CSV

MaveDB provides score data as CSV through the `/scores` endpoint.

In [11]:
scores_response = requests.get(f"{BASE_URL}/score-sets/{score_set_urn}/scores", timeout=60)
scores_response.raise_for_status()

scores_path = DATA_DIR / f"{score_set_urn.replace(':', '_')}_scores.csv"
scores_path.write_text(scores_response.text, encoding="utf-8")

scores_path

PosixPath('data/mavedb/urn_mavedb_00000001-c-2_scores.csv')

## 7. Load the raw score table

In [12]:
df_raw = read_table(scores_path)
print(df_raw.shape)
df_raw.head()

(2980, 11)


,accession,hgvs_nt,hgvs_splice,hgvs_pro,score,sd,se,exp.score,exp.sd,df,pred.score
0,urn:mavedb:00000001-c-2#1,NaN,NaN,p.Ser102Ala,0.435840,0.320421,0.160210,0.399769,0.255159,4.0,0.567830
1,urn:mavedb:00000001-c-2#2,NaN,NaN,p.Thr6Met,1.128538,0.172833,0.086416,1.149961,0.085079,4.0,0.893434
2,urn:mavedb:00000001-c-2#3,NaN,NaN,p.Glu83Cys,-0.007668,0.329071,0.164536,-0.041031,0.267999,4.0,0.108565
3,urn:mavedb:00000001-c-2#4,NaN,NaN,p.Thr147Gln,0.836352,0.466839,0.466839,NaN,NaN,NaN,0.836352
4,urn:mavedb:00000001-c-2#5,NaN,NaN,p.Val122Ile,0.956845,0.079884,0.039942,0.957673,0.025238,4.0,0.926240


## 8. Inspect columns

MaveDB score tables usually contain uploaded score/count columns plus MaveDB-generated columns.
For our substitution-only workflow, we are mainly interested in:
- `hgvs_pro`
- a score column

In [13]:
print(df_raw.columns.tolist())

['accession', 'hgvs_nt', 'hgvs_splice', 'hgvs_pro', 'score', 'sd', 'se', 'exp.score', 'exp.sd', 'df', 'pred.score']


## 9. Detect candidate columns

Score columns are dataset-specific, so we detect them conservatively.

In [14]:
candidate_hgvs_cols = ["hgvs_pro"]
candidate_score_cols = ["score", "DMS_score", "fitness"]

hgvs_col = next((c for c in candidate_hgvs_cols if c in df_raw.columns), None)
score_col = next((c for c in candidate_score_cols if c in df_raw.columns), None)

print("Detected hgvs column:", hgvs_col)
print("Detected score column:", score_col)

if hgvs_col is None:
    raise ValueError("No hgvs_pro column was found in this score set.")

if score_col is None:
    raise ValueError("Could not automatically detect a score column.")

Detected hgvs column: hgvs_pro
Detected score column: score


## 10. Recover or define the WT sequence

For substitution-only reconstruction we need the wild-type protein sequence.

Depending on the score-set metadata, the target sequence may or may not be directly available in the JSON payload.
If it is not available, provide it manually.

In [17]:
def find_sequence_fields(obj):
    """Collect candidate sequence-like fields from nested metadata."""
    hits = []

    def _walk(x, path="root"):
        if isinstance(x, dict):
            for k, v in x.items():
                _walk(v, f"{path}.{k}")
        elif isinstance(x, list):
            for i, v in enumerate(x):
                _walk(v, f"{path}[{i}]")
        elif isinstance(x, str):
            hits.append((path, x.strip()))

    _walk(obj)
    return hits


all_string_fields = find_sequence_fields(score_set_meta)

sequence_like_fields = [
    (path, value)
    for path, value in all_string_fields
    if "sequence" in path.lower()
]

sequence_like_fields[:20]

[('root.targetGenes[0].targetSequence.sequenceType', 'dna'),
 ('root.targetGenes[0].targetSequence.sequence',
  'ATGGCTGATCAGCTGACCGAAGAACAGATTGCTGAATTCAAGGAAGCCTTCTCCCTATTTGATAAAGATGGCGATGGCACCATCACAACAAAGGAACTTGGAACTGTCATGAGGTCACTGGGTCAGAACCCAACAGAAGCTGAATTGCAGGATATGATCAATGAAGTGGATGCTGATGGTAATGGCACCATTGACTTCCCCGAATTTTTGACTATGATGGCTAGAAAAATGAAAGATACAGATAGTGAAGAAGAAATCCGTGAGGCATTCCGAGTCTTTGACAAGGATGGCAATGGTTATATCAGTGCAGCAGAACTACGTCACGTCATGACAAACTTAGGAGAAAAACTAACAGATGAAGAAGTAGATGAAATGATCAGAGAAGCAGATATTGATGGAGACGGACAAGTCAACTATGAAGAATTCGTACAGATGATGACTGCAAAATGA'),
 ('root.targetGenes[0].targetSequence.recordType', 'TargetSequence'),
 ('root.targetGenes[0].targetSequence.taxonomy.organismName', 'Homo sapiens'),
 ('root.targetGenes[0].targetSequence.taxonomy.commonName', 'human'),
 ('root.targetGenes[0].targetSequence.taxonomy.rank', 'SPECIES'),
 ('root.targetGenes[0].targetSequence.taxonomy.articleReference',
  'NCBI:txid9606'),
 ('root.targetGenes[0].targetSequence.taxonomy.recordType', 'T

## 11. Set the WT sequence

If the metadata search above finds the correct protein sequence, use it.
Otherwise, paste the WT sequence manually below.

In [18]:
from dms_parser import translate_dna


def looks_like_dna(seq: str) -> bool:
    seq = seq.strip().upper()
    if not seq:
        return False
    return set(seq) <= set("ACGTN")


def looks_like_protein(seq: str) -> bool:
    seq = seq.strip().upper()
    if not seq:
        return False
    return set(seq) <= set("ACDEFGHIKLMNPQRSTVWYBXZJUO*")


wt_sequence = None
wt_sequence_source = None
wt_sequence_type = None

for path, seq in sequence_like_fields:
    seq_clean = seq.strip().upper()

    # Prioritize fields explicitly named targetSequence.sequence
    if path.lower().endswith("targetsequence.sequence"):
        wt_sequence_source = path

        if looks_like_dna(seq_clean):
            wt_sequence = translate_dna(seq_clean, frame=1, stop_at_stop=True)
            wt_sequence_type = "DNA translated to protein"
        elif looks_like_protein(seq_clean):
            wt_sequence = seq_clean
            wt_sequence_type = "protein"
        break

# fallback: use any other sequence-like field if needed
if wt_sequence is None:
    for path, seq in sequence_like_fields:
        seq_clean = seq.strip().upper()

        if looks_like_protein(seq_clean):
            wt_sequence = seq_clean
            wt_sequence_source = path
            wt_sequence_type = "protein"
            break

        if looks_like_dna(seq_clean):
            wt_sequence = translate_dna(seq_clean, frame=1, stop_at_stop=True)
            wt_sequence_source = path
            wt_sequence_type = "DNA translated to protein"
            break

if wt_sequence is None:
    raise ValueError(
        "Could not recover a valid WT sequence automatically from score set metadata. "
        "Please provide it manually."
    )

print("WT source:", wt_sequence_source)
print("WT type:", wt_sequence_type)
print("WT length:", len(wt_sequence))
print("WT preview:", wt_sequence[:60])

WT source: root.targetGenes[0].targetSequence.sequence
WT type: DNA translated to protein
WT length: 149
WT preview: MADQLTEEQIAEFKEAFSLFDKDGDGTITTKELGTVMRSLGQNPTEAELQDMINEVDADG


In [19]:
from dms_parser.validation import validate_wt_sequence

validate_wt_sequence(wt_sequence)

## 12. Build the standardized dataset

In [20]:
df_built = build_mavedb_dataset(
    input_path=scores_path,
    score_col=score_col,
    hgvs_col=hgvs_col,
    dataset_id=score_set_urn,
    wt_sequence=wt_sequence,
    add_relative_score=False,   # many MaveDB score tables do not include an explicit WT row
    drop_failed=False,
)

df_built.head()

,accession,hgvs_nt,hgvs_splice,hgvs_pro,score,sd,se,exp.score,exp.sd,df,...,gene,uniprot_id,wt_sequence,variant,mutated_sequence,status,error,is_wildtype,n_mutations,score_raw
0,urn:mavedb:00000001-c-2#1,NaN,NaN,p.Ser102Ala,0.435840,0.320421,0.160210,0.399769,0.255159,4.0,...,None,None,MADQLTEEQIAEFKEAFSLFDKDGDGTITTKELGTVMRSLGQNPTE...,S102A,MADQLTEEQIAEFKEAFSLFDKDGDGTITTKELGTVMRSLGQNPTE...,OK,,False,1,0.435840
1,urn:mavedb:00000001-c-2#2,NaN,NaN,p.Thr6Met,1.128538,0.172833,0.086416,1.149961,0.085079,4.0,...,None,None,MADQLTEEQIAEFKEAFSLFDKDGDGTITTKELGTVMRSLGQNPTE...,T6M,MADQLMEEQIAEFKEAFSLFDKDGDGTITTKELGTVMRSLGQNPTE...,OK,,False,1,1.128538
2,urn:mavedb:00000001-c-2#3,NaN,NaN,p.Glu83Cys,-0.007668,0.329071,0.164536,-0.041031,0.267999,4.0,...,None,None,MADQLTEEQIAEFKEAFSLFDKDGDGTITTKELGTVMRSLGQNPTE...,E83C,MADQLTEEQIAEFKEAFSLFDKDGDGTITTKELGTVMRSLGQNPTE...,OK,,False,1,-0.007668
3,urn:mavedb:00000001-c-2#4,NaN,NaN,p.Thr147Gln,0.836352,0.466839,0.466839,NaN,NaN,NaN,...,None,None,MADQLTEEQIAEFKEAFSLFDKDGDGTITTKELGTVMRSLGQNPTE...,T147Q,MADQLTEEQIAEFKEAFSLFDKDGDGTITTKELGTVMRSLGQNPTE...,OK,,False,1,0.836352
4,urn:mavedb:00000001-c-2#5,NaN,NaN,p.Val122Ile,0.956845,0.079884,0.039942,0.957673,0.025238,4.0,...,None,None,MADQLTEEQIAEFKEAFSLFDKDGDGTITTKELGTVMRSLGQNPTE...,V122I,MADQLTEEQIAEFKEAFSLFDKDGDGTITTKELGTVMRSLGQNPTE...,OK,,False,1,0.956845


## 13. Inspect standardized output

In [22]:
cols = [
    "variant",
    "score_raw",
    "status",
    "error",
    "is_wildtype",
    "n_mutations",
    "mutated_sequence",
]

available_cols = [c for c in cols if c in df_built.columns]
df_built[available_cols].head(5)

,variant,score_raw,status,error,is_wildtype,n_mutations,mutated_sequence
0,S102A,0.435840,OK,,False,1,MADQLTEEQIAEFKEAFSLFDKDGDGTITTKELGTVMRSLGQNPTE...
1,T6M,1.128538,OK,,False,1,MADQLMEEQIAEFKEAFSLFDKDGDGTITTKELGTVMRSLGQNPTE...
2,E83C,-0.007668,OK,,False,1,MADQLTEEQIAEFKEAFSLFDKDGDGTITTKELGTVMRSLGQNPTE...
3,T147Q,0.836352,OK,,False,1,MADQLTEEQIAEFKEAFSLFDKDGDGTITTKELGTVMRSLGQNPTE...
4,V122I,0.956845,OK,,False,1,MADQLTEEQIAEFKEAFSLFDKDGDGTITTKELGTVMRSLGQNPTE...


## 14. Summarize parsing status

MaveDB datasets often include unsupported protein-level events such as indels, duplications, frameshifts, or extensions.
For a substitutions-only workflow, these should remain flagged instead of silently discarded.

In [23]:
df_built["status"].value_counts(dropna=False)

status
OK    2980
Name: count, dtype: int64

In [24]:
df_built[df_built["status"] != "OK"][["variant", "status", "error"]].head(20)

,variant,status,error


## 15. Keep only successfully parsed substitution rows

In [25]:
df_ok = df_built[df_built["status"] == "OK"].copy()

print("All rows:", len(df_built))
print("OK rows:", len(df_ok))
df_ok.head()

All rows: 2980
OK rows: 2980


,accession,hgvs_nt,hgvs_splice,hgvs_pro,score,sd,se,exp.score,exp.sd,df,...,gene,uniprot_id,wt_sequence,variant,mutated_sequence,status,error,is_wildtype,n_mutations,score_raw
0,urn:mavedb:00000001-c-2#1,NaN,NaN,p.Ser102Ala,0.435840,0.320421,0.160210,0.399769,0.255159,4.0,...,None,None,MADQLTEEQIAEFKEAFSLFDKDGDGTITTKELGTVMRSLGQNPTE...,S102A,MADQLTEEQIAEFKEAFSLFDKDGDGTITTKELGTVMRSLGQNPTE...,OK,,False,1,0.435840
1,urn:mavedb:00000001-c-2#2,NaN,NaN,p.Thr6Met,1.128538,0.172833,0.086416,1.149961,0.085079,4.0,...,None,None,MADQLTEEQIAEFKEAFSLFDKDGDGTITTKELGTVMRSLGQNPTE...,T6M,MADQLMEEQIAEFKEAFSLFDKDGDGTITTKELGTVMRSLGQNPTE...,OK,,False,1,1.128538
2,urn:mavedb:00000001-c-2#3,NaN,NaN,p.Glu83Cys,-0.007668,0.329071,0.164536,-0.041031,0.267999,4.0,...,None,None,MADQLTEEQIAEFKEAFSLFDKDGDGTITTKELGTVMRSLGQNPTE...,E83C,MADQLTEEQIAEFKEAFSLFDKDGDGTITTKELGTVMRSLGQNPTE...,OK,,False,1,-0.007668
3,urn:mavedb:00000001-c-2#4,NaN,NaN,p.Thr147Gln,0.836352,0.466839,0.466839,NaN,NaN,NaN,...,None,None,MADQLTEEQIAEFKEAFSLFDKDGDGTITTKELGTVMRSLGQNPTE...,T147Q,MADQLTEEQIAEFKEAFSLFDKDGDGTITTKELGTVMRSLGQNPTE...,OK,,False,1,0.836352
4,urn:mavedb:00000001-c-2#5,NaN,NaN,p.Val122Ile,0.956845,0.079884,0.039942,0.957673,0.025238,4.0,...,None,None,MADQLTEEQIAEFKEAFSLFDKDGDGTITTKELGTVMRSLGQNPTE...,V122I,MADQLTEEQIAEFKEAFSLFDKDGDGTITTKELGTVMRSLGQNPTE...,OK,,False,1,0.956845


## 16. Summary

In this notebook we:
- searched MaveDB programmatically
- selected a real score set by URN
- downloaded its score table as CSV
- recovered or provided the WT sequence
- converted the score set into a standardized substitutions-only dataset with `dms_parser`